# Exploratory Data Analysis

In [0]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, ArrayType, MapType

In [0]:
file_type = 'json'
file_location = '/Volumes/workspace/it3388/raw_data/game_metadata/games.json'

metadata_df = pd.read_json(file_location)
metadata_df = metadata_df.T.reset_index()
metadata_df = metadata_df.astype(str)
metadata_df = metadata_df.rename(columns={'index': 'app_id'})
display(metadata_df)

In [0]:
spark_df = spark.createDataFrame(metadata_df)
spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.it3388.bronze_game_metadata")

In [0]:
metadata_df = spark.table("workspace.it3388.bronze_game_metadata")
metadata_df = metadata_df[["app_id", "name", "release_date", "price", "dlc_count", "windows", "mac", "linux", "achievements", "recommendations", "supported_languages", "full_audio_languages", "developers", "publishers", "categories", "genres", "positive", "negative", "estimated_owners", "average_playtime_forever", "median_playtime_forever", "peak_ccu", "tags"]]

print(f"Number of rows: {metadata_df.count()}")
print(f"Number of columns: {len(metadata_df.columns)}")
print(f"Column names: {metadata_df.columns}")

In [0]:
duplicate_count = metadata_df.groupBy(metadata_df.columns).count().filter("count > 1").count()
print(f"Number of duplicate rows: {duplicate_count}")

In [0]:
from pyspark.sql.functions import count, when, col

null_counts = metadata_df.select([count(when(col(c).isNull(), c)).alias(c) for c in metadata_df.columns])
display(null_counts)

In [0]:
# Create a summary of empty array counts
total_rows = metadata_df.count()

empty_array_counts = metadata_df.select(
    [count(when(col(c) == '[]', c)).alias(c) for c in metadata_df.columns]
).collect()[0].asDict()

# Convert to pandas for visualization
summary_df = pd.DataFrame([
    {'column': col, 'empty_array_count': count, 'completeness_pct': round((1 - count/total_rows) * 100, 2)}
    for col, count in empty_array_counts.items()
    if count > 0
]).sort_values('empty_array_count', ascending=False)
display(summary_df)

# Create visualization
fig, ax = plt.subplots(figsize=(12, 6))
summary_df.plot(x='column', y='completeness_pct', kind='bar', ax=ax, color='steelblue')
ax.set_ylabel('Data Completeness (%)')
ax.set_xlabel('Column')
ax.set_title('Data Completeness by Column (100% = No Empty Arrays)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Data cleaning and preparation

In [0]:
from pyspark.sql.functions import col, to_date, regexp_replace, from_json, round
from pyspark.sql.types import ArrayType, StringType, MapType, IntegerType, FloatType, DoubleType, BooleanType

# Parse columns to correct data types
cleaned_metadata_df = metadata_df \
    .withColumn("release_date", to_date(col("release_date"), "MMM d, yyyy")) \
    .withColumn("price", col("price").cast(DoubleType())) \
    .withColumn("dlc_count", col("dlc_count").cast(IntegerType())) \
    .withColumn("windows", col("windows").cast(BooleanType())) \
    .withColumn("mac", col("mac").cast(BooleanType())) \
    .withColumn("linux", col("linux").cast(BooleanType())) \
    .withColumn("achievements", col("achievements").cast(IntegerType())) \
    .withColumn("recommendations", col("recommendations").cast(IntegerType())) \
    .withColumn("supported_languages", from_json(col("supported_languages"), ArrayType(StringType()))) \
    .withColumn("full_audio_languages", from_json(col("full_audio_languages"), ArrayType(StringType()))) \
    .withColumn("developers", from_json(col("developers"), ArrayType(StringType()))) \
    .withColumn("publishers", from_json(col("publishers"), ArrayType(StringType()))) \
    .withColumn("categories", from_json(col("categories"), ArrayType(StringType()))) \
    .withColumn("genres", from_json(col("genres"), ArrayType(StringType()))) \
    .withColumn("positive", col("positive").cast(IntegerType())) \
    .withColumn("negative", col("negative").cast(IntegerType())) \
    .withColumn("average_playtime_forever", col("average_playtime_forever").cast(DoubleType())) \
    .withColumn("median_playtime_forever", col("median_playtime_forever").cast(DoubleType())) \
    .withColumn("tags", from_json(col("tags"), MapType(StringType(), IntegerType()))) \
    .sort(col("app_id").asc())
    
display(cleaned_metadata_df)

In [0]:
# Get unique languages from the supported_languages array column
supported_langs = cleaned_metadata_df.selectExpr("explode(supported_languages) as lang").distinct().collect()
print([row.lang for row in supported_langs])

In [0]:
# Get unique languages from the full_audio_languages array column
audio_langs = cleaned_metadata_df.selectExpr("explode(full_audio_languages) as lang").distinct().collect()
print([row.lang for row in audio_langs])

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType
import re

def clean_language_list(lang_array):
    if not lang_array:
        return []
    
    cleaned_languages = set()
    for lang in lang_array:
        if lang and isinstance(lang, str):
            for separator in ['\r\n', ',', ';']:
                if separator in lang:
                    lang = lang.replace(separator, '|')
            
            # Split into individual languages
            langs = [l.strip() for l in lang.split('|')]
            
            for l in langs:
                if l:
                    l = re.sub(r'\([^)]*\)', '', l)
                    l = re.sub(r'&[^;\s]+;', '', l)
                    l = re.sub(r'\[/?b\]', '', l)
                    l = re.sub(r'<[^>]+>', '', l)
                    l = re.sub(r'\b(amp|lt|gt|br|strong|/strong)\b', '', l)
                    l = re.sub(r'[&;]+$', '', l)
                    l = re.sub(r'\s+', ' ', l)
                    l = l.strip()
                    words = l.split()
                    if len(words) == 3 and words[0] == words[2]:
                        l = words[0]
                    if l and not l.startswith('#') and not l.startswith('/') and len(l) > 1:
                        if re.search(r'[a-zA-Z]{2,}', l):
                            cleaned_languages.add(l)

    return sorted(list(cleaned_languages))

# Register UDF
clean_languages_udf = udf(clean_language_list, ArrayType(StringType()))

# Apply cleaning to both language columns
cleaned_metadata_df = cleaned_metadata_df \
    .withColumn("supported_languages", clean_languages_udf(col("supported_languages"))) \
    .withColumn("full_audio_languages", clean_languages_udf(col("full_audio_languages")))

# Show sample of cleaned data
display(cleaned_metadata_df.limit(10))

In [0]:
# Extract all unique game categories from the dataset
categories = cleaned_metadata_df.selectExpr("explode(categories) as category").distinct().collect()
print([row.category for row in categories])

In [0]:
# Extract all unique game genres from the dataset
genres = cleaned_metadata_df.selectExpr("explode(genres) as genre").distinct().collect()
print([row.genre for row in genres])

In [0]:
from pyspark.sql.functions import array_contains, lit

# Extract unique values from array columns
supported_langs_cleaned = cleaned_metadata_df.selectExpr("explode(supported_languages) as lang").distinct().collect()
audio_langs_cleaned = cleaned_metadata_df.selectExpr("explode(full_audio_languages) as lang").distinct().collect()
categories_cleaned = cleaned_metadata_df.selectExpr("explode(categories) as cat").distinct().collect()
genres_cleaned = cleaned_metadata_df.selectExpr("explode(genres) as genre").distinct().collect()

# Create sorted lists
supported_languages_list = sorted([row.lang for row in supported_langs_cleaned])
audio_languages_list = sorted([row.lang for row in audio_langs_cleaned])
categories_list = sorted([row.cat for row in categories_cleaned])
genres_list = sorted([row.genre for row in genres_cleaned])

def format_column_name(prefix, value):
    cleaned = value.replace('(', '').replace(')', '').replace("'", '').replace('/', ' ').replace('&', 'and')
    cleaned = cleaned.replace('-', ' ')
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    cleaned = cleaned.replace(' ', '_')
    cleaned = re.sub(r'_+', '_', cleaned)
    cleaned = cleaned.strip('_')
    return f"{prefix}_{cleaned}"

new_columns = {}

# Create columns for supported_languages
for lang in supported_languages_list:
    col_name = format_column_name("supported_languages", lang)
    new_columns[col_name] = array_contains(col("supported_languages"), lang)

# Create columns for full_audio_languages
for lang in audio_languages_list:
    col_name = format_column_name("full_audio_languages", lang)
    new_columns[col_name] = array_contains(col("full_audio_languages"), lang)

# Create columns for categories
for cat in categories_list:
    col_name = format_column_name("categories", cat)
    new_columns[col_name] = array_contains(col("categories"), cat)

# Create columns for genres
for genre in genres_list:
    col_name = format_column_name("genres", genre)
    new_columns[col_name] = array_contains(col("genres"), genre)

# Apply all columns at once
cleaned_metadata_df = cleaned_metadata_df.withColumns(new_columns)

# Show samples
display(cleaned_metadata_df.limit(10))

In [0]:
from pyspark.sql.functions import explode, count as spark_count, desc

# Get unique developers and publishers with their game counts
developers_count = cleaned_metadata_df.select(explode(col("developers")).alias("developer")) \
    .groupBy("developer") \
    .agg(spark_count("*").alias("game_count")) \
    .orderBy(desc("game_count"))

publishers_count = cleaned_metadata_df.select(explode(col("publishers")).alias("publisher")) \
    .groupBy("publisher") \
    .agg(spark_count("*").alias("game_count")) \
    .orderBy(desc("game_count"))

print(f"Total unique developers: {developers_count.count()}")
print(f"Total unique publishers: {publishers_count.count()}")
print(f"\nTop 20 developers by game count:")
display(developers_count.limit(20))
print(f"\nTop 20 publishers by game count:")
display(publishers_count.limit(20))

In [0]:
from pyspark.sql.functions import explode, map_keys, map_values, sum

# Extract all unique tag names and aggregate their total counts across all games
tags_exploded = cleaned_metadata_df.select(
    explode(col("tags")).alias("tag_name", "tag_count")
)

# Aggregate by tag name to get total count and number of games with this tag
tags_summary = tags_exploded.groupBy("tag_name") \
    .agg(
        spark_count("*").alias("num_games"),
        sum(col("tag_count")).alias("total_votes")
    ) \
    .orderBy(desc("num_games"))

print(f"Total unique tags: {tags_summary.count()}")
print(f"\nTop 30 tags by number of games:")
display(tags_summary.limit(30))

In [0]:
from pyspark.sql.functions import coalesce, element_at
import re

tags_exploded = cleaned_metadata_df.select(
    explode(col("tags")).alias("tag_name", "tag_count")
)

tags_list = sorted([row.tag_name for row in tags_exploded.select("tag_name").distinct().collect()])
tag_columns = {}

for tag in tags_list:
    col_name = format_column_name("tags", tag)
    # Extract the count value for this tag from the map, default to 0 if tag doesn't exist
    tag_columns[col_name] = coalesce(element_at(col("tags"), lit(tag)), lit(0))

cleaned_metadata_df = cleaned_metadata_df.withColumns(tag_columns)
display(cleaned_metadata_df.limit(10))

In [0]:
cleaned_metadata_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.it3388.silver_game_metadata")

In [0]:
# Distribution of games by Indie genre
import matplotlib.pyplot as plt
from pyspark.sql.functions import col, count as spark_count

metadata_df = spark.table("workspace.it3388.silver_game_metadata")

# Count games by indie status
indie_distribution = metadata_df.groupBy("genres_Indie") \
    .agg(spark_count("*").alias("game_count")) \
    .orderBy("genres_Indie")

# Convert to pandas for visualization
indie_pd = indie_distribution.toPandas()

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#3498db', '#e74c3c']
labels = ['Non-Indie', 'Indie']
ax1.bar(labels, indie_pd['game_count'], color=colors)
ax1.set_ylabel('Number of Games')
ax1.set_title('Distribution of Games by Indie Genre')
ax1.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, v in enumerate(indie_pd['game_count']):
    ax1.text(i, v + 500, str(v), ha='center', va='bottom', fontweight='bold')

# Pie chart
ax2.pie(indie_pd['game_count'], labels=labels, colors=colors, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
ax2.set_title('Percentage Distribution by Indie Genre')

plt.tight_layout()
plt.show()